# Indicadores de Muestras por Cuadrante - Manizales 2021-2024

## Análisis con IDBA Poisson-Gamma

**Objetivo:** Construir indicadores por cuadrante (2021-2024) con conteos, días de actividad, densidades naive e IDBA (Empirical Bayes robusto).

**Archivos de entrada:**
- `muestras_MANIZALES_2021-2024.csv`: Datos de muestras con columnas id, fecha_evento, id_autor, lat, lot, cod_cuadrante
- `manizales_metricas.csv`: Métricas geométricas con cod_cuadrante y area_m2

**Archivos de salida:**
- `indicadores_cuadrantes_MANIZALES_2021-2024.csv`: Dataset final con todos los indicadores
- `idba_priors_MANIZALES_2021-2024.csv`: Parámetros Bayesianos por año

**Modelo IDBA:** $k|ρ,A \sim \text{Poisson}(ρA)$, $ρ \sim \text{Gamma}(α,β)$ → $E[ρ|k,A] = \frac{α+k}{β+A}$

In [30]:
# ================================================================================================
# 1. IMPORTS Y CONFIGURACIÓN
# ================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 50)

# Suprimir warnings no críticos
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# Años de análisis
ANIOS = [2021, 2022, 2023, 2024]

# Escala para normalización (tasas por 1000 m²)  
SCALE = 1000

# Configuración de archivos (mismo directorio del notebook)
BASE_DIR = Path('.')
ARCHIVO_MUESTRAS = BASE_DIR / 'muestras_MANIZALES_2021-2024.csv'
ARCHIVO_METRICAS = BASE_DIR / 'manizales_metricas.csv'
ARCHIVO_SALIDA_INDICADORES = BASE_DIR / 'indicadores_cuadrantes_MANIZALES_2021-2024.csv'
ARCHIVO_SALIDA_PRIORS = BASE_DIR / 'idba_priors_MANIZALES_2021-2024.csv'

print("✅ Configuración completada")
print(f"📊 Años de análisis: {ANIOS}")
print(f"📏 Escala de normalización: {SCALE} (por 1000 m²)")
print(f"📁 Directorio base: {BASE_DIR.absolute()}")

✅ Configuración completada
📊 Años de análisis: [2021, 2022, 2023, 2024]
📏 Escala de normalización: 1000 (por 1000 m²)
📁 Directorio base: c:\Users\ESP_NEGOCIO\Documents\GitHub\MAPAS_TA_DEV_1\tools


In [31]:
# ================================================================================================
# 2. CARGA Y PREPARACIÓN DE DATOS
# ================================================================================================

print("📂 Cargando datos...")

# =============================================================================
# CARGA DE DATOS CON PARÁMETROS ESPECÍFICOS
# =============================================================================

# Cargar datos de muestras (sep=';', encoding='utf-8-sig')
print(f"   Leyendo muestras desde: {ARCHIVO_MUESTRAS}")
df_m = pd.read_csv(ARCHIVO_MUESTRAS, sep=';', encoding='utf-8-sig')

# Cargar métricas de cuadrantes (sep=';', encoding='utf-8-sig')  
print(f"   Leyendo métricas desde: {ARCHIVO_METRICAS}")
df_q = pd.read_csv(ARCHIVO_METRICAS, encoding='utf-8-sig')

print(f"📊 Muestras cargadas: {len(df_m):,} registros")
print(f"📐 Cuadrantes cargados: {len(df_q):,} cuadrantes")

print(df_m.columns)
print(df_q.columns)

# =============================================================================
# PREPARACIÓN DE FECHAS Y AÑOS
# =============================================================================

# Convertir fecha_evento a datetime
df_m['fecha_evento'] = pd.to_datetime(df_m['fecha_evento'], errors='coerce')

# Crear columna ANIO
df_m['ANIO'] = df_m['fecha_evento'].dt.year

# Filtrar solo años de interés y remover fechas inválidas
df_m_original = df_m.copy()
df_m = df_m[df_m['ANIO'].isin(ANIOS) & df_m['fecha_evento'].notna()]

print(f"📅 Registros válidos en período {min(ANIOS)}-{max(ANIOS)}: {len(df_m):,}")
print(f"🗑️ Registros filtrados: {len(df_m_original) - len(df_m):,}")

# =============================================================================
# NORMALIZACIÓN DE CÓDIGOS DE CUADRANTE
# =============================================================================

# Normalizar códigos en muestras: str → upper → strip → fillna('FUERA')
df_m['cod_cuadrante'] = (df_m['cod_cuadrante']
                        .astype(str)
                        .str.upper()
                        .str.strip()
                        .replace('NAN', np.nan)  # Convertir 'nan' string a NaN real
                        .fillna('FUERA'))

# Normalizar códigos en métricas (renombrar si necesario)
if 'codigo' in df_q.columns and 'cod_cuadrante' not in df_q.columns:
    df_q = df_q.rename(columns={'codigo': 'cod_cuadrante'})
    print("🔄 Renombrado 'codigo' → 'cod_cuadrante' en métricas")

df_q['cod_cuadrante'] = df_q['cod_cuadrante'].astype(str).str.upper().str.strip()

# =============================================================================
# PREPARACIÓN DE ÁREAS
# =============================================================================

# Validar y preparar columna de área
if 'area_m2' not in df_q.columns:
    if 'area' in df_q.columns:
        print("🔄 Convirtiendo 'area' → 'area_m2'")
        # Detectar si está en km² (valores típicamente < 100) vs m² (valores > 1000)
        sample_area = df_q['area'].dropna().iloc[:10] if len(df_q['area'].dropna()) > 0 else pd.Series([1])
        if sample_area.mean() < 100:  # Probablemente km²
            df_q['area_m2'] = df_q['area'] * 1e6  # km² → m²
            print("   Detectado área en km², convertido a m²")
        else:
            df_q['area_m2'] = df_q['area']  # Ya en m²
            print("   Área ya en m², copiado directamente")
    else:
        raise ValueError("No se encontró columna de área (area_m2 o area)")

# Crear area_km2 para legibilidad (solo informativa)
df_q['area_km2'] = df_q['area_m2'] / 1e6

print(f"📏 Área promedio: {df_q['area_m2'].mean():,.0f} m² ({df_q['area_km2'].mean():.3f} km²)")
print(f"📏 Área mediana: {df_q['area_m2'].median():,.0f} m² ({df_q['area_km2'].median():.3f} km²)")

# =============================================================================
# RESUMEN DE DATOS PREPARADOS
# =============================================================================

print("\n🎯 RESUMEN DE PREPARACIÓN:")
print(f"   • Muestras por año: {df_m.groupby('ANIO').size().to_dict()}")
print(f"   • Cuadrantes únicos en muestras: {df_m['cod_cuadrante'].nunique()}")
print(f"   • Cuadrantes con 'FUERA': {(df_m['cod_cuadrante'] == 'FUERA').sum():,}")
print(f"   • Cuadrantes en métricas: {len(df_q)}")
print("✅ Datos preparados correctamente")

📂 Cargando datos...
   Leyendo muestras desde: muestras_MANIZALES_2021-2024.csv
   Leyendo métricas desde: manizales_metricas.csv
📊 Muestras cargadas: 26,977 registros
📐 Cuadrantes cargados: 30 cuadrantes
Index(['id', 'fecha_evento', 'id_autor', 'lat', 'lot', 'cod_cuadrante'], dtype='object')
Index(['cod_cuadrante', 'ciudad', 'area_m2', 'perimetro_m', 'centroid_lon', 'centroid_lat', 'compactness_polsby',
       'elongation_ratio', 'holes_count'],
      dtype='object')
📅 Registros válidos en período 2021-2024: 26,977
🗑️ Registros filtrados: 0
📏 Área promedio: 455,453 m² (0.455 km²)
📏 Área mediana: 489,497 m² (0.489 km²)

🎯 RESUMEN DE PREPARACIÓN:
   • Muestras por año: {2021: 8138, 2022: 21, 2023: 8567, 2024: 10251}
   • Cuadrantes únicos en muestras: 31
   • Cuadrantes con 'FUERA': 1,108
   • Cuadrantes en métricas: 30
✅ Datos preparados correctamente


In [32]:
# ================================================================================================
# 3. AGREGACIONES BASE POR CUADRANTE Y AÑO  
# ================================================================================================

print("🔢 Calculando conteos por cuadrante y año...")

# =============================================================================
# CONTEOS POR AÑO USANDO PIVOT TABLE
# =============================================================================

# Crear tabla de conteos: filas=cod_cuadrante, columnas=ANIO, valores=count
conteos_pivot = (df_m.groupby(['cod_cuadrante', 'ANIO'])
                .size()
                .reset_index(name='count')
                .pivot(index='cod_cuadrante', columns='ANIO', values='count')
                .fillna(0)  # Rellenar con 0 donde no hay datos
                .astype(int))

# Renombrar columnas a formato k_YYYY
conteos_pivot.columns = [f'k_{int(year)}' for year in conteos_pivot.columns]

# Asegurar que tenemos todas las columnas de años esperadas
for year in ANIOS:
    col_name = f'k_{year}'
    if col_name not in conteos_pivot.columns:
        conteos_pivot[col_name] = 0

# Reordenar columnas en orden cronológico
year_cols = [f'k_{year}' for year in sorted(ANIOS)]
conteos_pivot = conteos_pivot[year_cols]

print(f"📊 Cuadrantes con datos: {len(conteos_pivot)}")

# =============================================================================
# CÁLCULO DE TOTALES
# =============================================================================

# Calcular k_total = suma de todos los años
conteos_pivot['k_total'] = conteos_pivot[year_cols].sum(axis=1)

# Convertir índice a columna para facilitar merges posteriores
df_conteos = conteos_pivot.reset_index()

print(f"🎯 Total muestras procesadas: {df_conteos['k_total'].sum():,}")
print(f"📈 Distribución por año:")
for year in ANIOS:
    col = f'k_{year}'
    total = df_conteos[col].sum()
    print(f"   • {year}: {total:,} muestras")

# =============================================================================
# VERIFICAR CUADRANTE 'FUERA'
# =============================================================================

fuera_idx = df_conteos['cod_cuadrante'] == 'FUERA'
if fuera_idx.any():
    fuera_total = df_conteos.loc[fuera_idx, 'k_total'].iloc[0]
    print(f"🔍 Registros 'FUERA': {fuera_total:,} ({fuera_total/df_conteos['k_total'].sum()*100:.1f}%)")
else:
    print("ℹ️  No hay registros 'FUERA' (todos tienen cuadrante válido)")

print(f"\n📋 Primeras filas de conteos:")
print(df_conteos.to_string(index=False))
print("✅ Agregaciones base completadas")

🔢 Calculando conteos por cuadrante y año...
📊 Cuadrantes con datos: 31
🎯 Total muestras procesadas: 26,977
📈 Distribución por año:
   • 2021: 8,138 muestras
   • 2022: 21 muestras
   • 2023: 8,567 muestras
   • 2024: 10,251 muestras
🔍 Registros 'FUERA': 1,108 (4.1%)

📋 Primeras filas de conteos:
cod_cuadrante  k_2021  k_2022  k_2023  k_2024  k_total
        FUERA     222      21     395     470     1108
       MZ_001       0       0      62       0       62
       MZ_002       0       0     563       0      563
       MZ_003       1       0     588       0      589
       MZ_004       0       0       0     167      167
       MZ_005     131       0     279      88      498
       MZ_006       0       0       0      39       39
       MZ_007       0       0     226     988     1214
       MZ_008     362       0     199     231      792
       MZ_009       0       0     936    1055     1991
       MZ_010       0       0      58       0       58
       MZ_011       3       0      59      

In [33]:
# ================================================================================================
# 4. DÍAS CON ACTIVIDAD Y ≥2 MUESTRAS POR AÑO
# ================================================================================================

print("📅 Calculando días con actividad...")

# =============================================================================
# PREPARAR DATOS DIARIOS
# =============================================================================

# Crear columna FECHA (solo la parte de fecha, sin hora)
df_m['FECHA'] = df_m['fecha_evento'].dt.date

# Agrupar por cuadrante, año y fecha para contar muestras por día
daily_counts = (df_m.groupby(['cod_cuadrante', 'ANIO', 'FECHA'])
               .size()
               .reset_index(name='muestras_dia'))

print(f"📈 Días únicos con datos: {len(daily_counts):,}")

# =============================================================================
# CALCULAR DÍAS CON ACTIVIDAD (≥1 muestra)
# =============================================================================

# Días con al menos 1 muestra por cuadrante y año
dias_any = (daily_counts.groupby(['cod_cuadrante', 'ANIO'])
           .size()  # Cuenta días únicos
           .reset_index(name='dias_any')
           .pivot(index='cod_cuadrante', columns='ANIO', values='dias_any')
           .fillna(0)
           .astype(int))

# Renombrar columnas
dias_any.columns = [f'dias_any_{int(year)}' for year in dias_any.columns]

# Asegurar todas las columnas de años
for year in ANIOS:
    col_name = f'dias_any_{year}'
    if col_name not in dias_any.columns:
        dias_any[col_name] = 0

# =============================================================================
# CALCULAR DÍAS CON ≥2 MUESTRAS
# =============================================================================

# Filtrar días con 2+ muestras
daily_ge2 = daily_counts[daily_counts['muestras_dia'] >= 2]

# Contar días ≥2 por cuadrante y año
dias_ge2 = (daily_ge2.groupby(['cod_cuadrante', 'ANIO'])
           .size()
           .reset_index(name='dias_ge2')
           .pivot(index='cod_cuadrante', columns='ANIO', values='dias_ge2')
           .fillna(0)
           .astype(int))

# Renombrar columnas
if len(dias_ge2.columns) > 0:
    dias_ge2.columns = [f'dias_ge2_{int(year)}' for year in dias_ge2.columns]

# Asegurar todas las columnas de años
for year in ANIOS:
    col_name = f'dias_ge2_{year}'
    if col_name not in dias_ge2.columns:
        dias_ge2[col_name] = 0

# =============================================================================
# CALCULAR TOTALES 4 AÑOS
# =============================================================================

# Totales de días any
dias_any_cols = [f'dias_any_{year}' for year in ANIOS]
dias_any['dias_any_total'] = dias_any[dias_any_cols].sum(axis=1)

# Totales de días ≥2
dias_ge2_cols = [f'dias_ge2_{year}' for year in ANIOS]  
dias_ge2['dias_ge2_total'] = dias_ge2[dias_ge2_cols].sum(axis=1)

# =============================================================================
# MERGE DE DÍAS CON CONTEOS PARA CALCULAR PROMEDIOS
# =============================================================================

# Preparar dataframes para merge
dias_any_df = dias_any.reset_index()
dias_ge2_df = dias_ge2.reset_index()

# Merge con conteos
df_combined = df_conteos.merge(dias_any_df, on='cod_cuadrante', how='left')
df_combined = df_combined.merge(dias_ge2_df, on='cod_cuadrante', how='left')

# Rellenar NaN con 0 en días
dias_cols = [col for col in df_combined.columns if col.startswith('dias_')]
df_combined[dias_cols] = df_combined[dias_cols].fillna(0).astype(int)

# =============================================================================
# CALCULAR PROMEDIOS DIARIOS
# =============================================================================

# Promedios diarios por año (any activity)
for year in ANIOS:
    k_col = f'k_{year}'
    dias_col = f'dias_any_{year}'
    avg_col = f'avg_dia_any_{year}'
    
    # avg = k / dias, pero solo donde dias > 0
    df_combined[avg_col] = np.where(
        df_combined[dias_col] > 0,
        df_combined[k_col] / df_combined[dias_col],
        np.nan
    )

# Promedios diarios por año (≥2 activity)  
for year in ANIOS:
    k_col = f'k_{year}'
    dias_col = f'dias_ge2_{year}'
    avg_col = f'avg_dia_ge2_{year}'
    
    df_combined[avg_col] = np.where(
        df_combined[dias_col] > 0,
        df_combined[k_col] / df_combined[dias_col],
        np.nan
    )

# Promedios totales
df_combined['avg_dia_any_total'] = np.where(
    df_combined['dias_any_total'] > 0,
    df_combined['k_total'] / df_combined['dias_any_total'],
    np.nan
)

df_combined['avg_dia_ge2_total'] = np.where(
    df_combined['dias_ge2_total'] > 0,
    df_combined['k_total'] / df_combined['dias_ge2_total'],
    np.nan
)

# =============================================================================
# RESUMEN DE DÍAS
# =============================================================================

print("📊 Resumen de días con actividad:")
for year in ANIOS:
    total_dias_any = df_combined[f'dias_any_{year}'].sum()
    total_dias_ge2 = df_combined[f'dias_ge2_{year}'].sum()
    print(f"   • {year}: {total_dias_any:,} días (≥1), {total_dias_ge2:,} días (≥2)")

print(f"\n📈 Promedios diarios globales:")
for year in ANIOS:
    avg_any = df_combined[f'avg_dia_any_{year}'].mean()
    avg_ge2 = df_combined[f'avg_dia_ge2_{year}'].mean()  
    print(f"   • {year}: {avg_any:.2f} muestras/día (≥1), {avg_ge2:.2f} muestras/día (≥2)")

print("✅ Cálculos de días completados")

📅 Calculando días con actividad...
📈 Días únicos con datos: 1,300
📊 Resumen de días con actividad:
   • 2021: 154 días (≥1), 129 días (≥2)
   • 2022: 14 días (≥1), 4 días (≥2)
   • 2023: 620 días (≥1), 454 días (≥2)
   • 2024: 512 días (≥1), 413 días (≥2)

📈 Promedios diarios globales:
   • 2021: 63.70 muestras/día (≥1), 74.04 muestras/día (≥2)
   • 2022: 1.50 muestras/día (≥1), 5.25 muestras/día (≥2)
   • 2023: 17.19 muestras/día (≥1), 21.60 muestras/día (≥2)
   • 2024: 27.37 muestras/día (≥1), 30.37 muestras/día (≥2)
✅ Cálculos de días completados


In [34]:
# ================================================================================================
# 5. MERGE CON ÁREAS DE CUADRANTES
# ================================================================================================

print("📐 Incorporando áreas de cuadrantes...")

# =============================================================================
# PREPARAR TABLA DE ÁREAS
# =============================================================================

# Seleccionar columnas de área y remover duplicados
areas = df_q[['cod_cuadrante', 'area_m2', 'area_km2']].drop_duplicates('cod_cuadrante')

print(f"📏 Cuadrantes con área disponible: {len(areas):,}")
print(f"📏 Área total disponible: {areas['area_km2'].sum():.2f} km²")

# =============================================================================
# MERGE CON DATOS AGREGADOS (LEFT JOIN)
# =============================================================================

# Hacer left join para mantener todos los cuadrantes de los datos
df_final = df_combined.merge(areas, on='cod_cuadrante', how='left')

# Verificar resultado del merge
cuadrantes_sin_area = df_final['area_m2'].isna().sum()
cuadrantes_con_area = df_final['area_m2'].notna().sum()

print(f"🔗 Merge completado:")
print(f"   • Cuadrantes con área: {cuadrantes_con_area:,}")
print(f"   • Cuadrantes sin área: {cuadrantes_sin_area:,}")

# =============================================================================
# VERIFICAR CUADRANTE 'FUERA'
# =============================================================================

fuera_mask = df_final['cod_cuadrante'] == 'FUERA'
if fuera_mask.any():
    fuera_area = df_final.loc[fuera_mask, 'area_m2'].iloc[0]
    if pd.isna(fuera_area):
        print("✅ Cuadrante 'FUERA' correctamente sin área (NaN)")
    else:
        print(f"⚠️  Cuadrante 'FUERA' tiene área: {fuera_area} m²")

# =============================================================================
# ESTADÍSTICAS DE ÁREAS
# =============================================================================

areas_validas = df_final[df_final['area_m2'] > 0]['area_m2']
if len(areas_validas) > 0:
    print(f"\n📊 Estadísticas de áreas (cuadrantes válidos):")
    print(f"   • Mínima: {areas_validas.min():,.0f} m² ({areas_validas.min()/1e6:.4f} km²)")
    print(f"   • Mediana: {areas_validas.median():,.0f} m² ({areas_validas.median()/1e6:.3f} km²)")
    print(f"   • Máxima: {areas_validas.max():,.0f} m² ({areas_validas.max()/1e6:.3f} km²)")
    print(f"   • Promedio: {areas_validas.mean():,.0f} m² ({areas_validas.mean()/1e6:.3f} km²)")

# =============================================================================
# LISTAR CUADRANTES SIN ÁREA (PARA QA)
# =============================================================================

cuadrantes_sin_area_list = df_final[df_final['area_m2'].isna()]['cod_cuadrante'].tolist()
if cuadrantes_sin_area_list:
    print(f"\n⚠️  Cuadrantes sin área ({len(cuadrantes_sin_area_list)}):")
    for i, cod in enumerate(cuadrantes_sin_area_list[:10]):  # Mostrar máximo 10
        muestras = df_final[df_final['cod_cuadrante'] == cod]['k_total'].iloc[0]
        print(f"   • {cod}: {muestras} muestras")
    if len(cuadrantes_sin_area_list) > 10:
        print(f"   • ... y {len(cuadrantes_sin_area_list) - 10} más")

print("✅ Incorporación de áreas completada")

📐 Incorporando áreas de cuadrantes...
📏 Cuadrantes con área disponible: 30
📏 Área total disponible: 13.66 km²
🔗 Merge completado:
   • Cuadrantes con área: 30
   • Cuadrantes sin área: 1
✅ Cuadrante 'FUERA' correctamente sin área (NaN)

📊 Estadísticas de áreas (cuadrantes válidos):
   • Mínima: 14,340 m² (0.0143 km²)
   • Mediana: 489,497 m² (0.489 km²)
   • Máxima: 1,287,667 m² (1.288 km²)
   • Promedio: 455,453 m² (0.455 km²)

⚠️  Cuadrantes sin área (1):
   • FUERA: 1108 muestras
✅ Incorporación de áreas completada


In [35]:
# ================================================================================================
# 6. DENSIDADES NAÏVE (SIN SUAVIZADO)
# ================================================================================================

print("🎯 Calculando densidades naïve...")

# =============================================================================
# CALCULAR DENSIDADES SIMPLES
# =============================================================================

# Crear mask para cuadrantes con área válida
area_valida = (df_final['area_m2'].notna()) & (df_final['area_m2'] > 0)

# Inicializar columnas de densidad con NaN
df_final['density_naive_per_m2'] = np.nan
df_final['density_naive_per_km2'] = np.nan

# Calcular densidades solo para cuadrantes con área válida
df_final.loc[area_valida, 'density_naive_per_m2'] = (
    df_final.loc[area_valida, 'k_total'] / df_final.loc[area_valida, 'area_m2']
)

df_final.loc[area_valida, 'density_naive_per_km2'] = (
    df_final.loc[area_valida, 'k_total'] / df_final.loc[area_valida, 'area_km2']
)

# =============================================================================
# ESTADÍSTICAS DE DENSIDADES NAÏVE
# =============================================================================

densidades_validas = df_final[area_valida]['density_naive_per_km2']
n_cuadrantes_con_densidad = len(densidades_validas)

print(f"📊 Densidades calculadas para {n_cuadrantes_con_densidad:,} cuadrantes")

if n_cuadrantes_con_densidad > 0:
    print(f"\n📈 Estadísticas de densidad naïve (muestras/km²):")
    print(f"   • Mínima: {densidades_validas.min():.4f}")
    print(f"   • Cuartil 1: {densidades_validas.quantile(0.25):.3f}")
    print(f"   • Mediana: {densidades_validas.median():.3f}")
    print(f"   • Cuartil 3: {densidades_validas.quantile(0.75):.3f}")
    print(f"   • Máxima: {densidades_validas.max():.2f}")
    print(f"   • Promedio: {densidades_validas.mean():.3f}")
    print(f"   • Desv. Estándar: {densidades_validas.std():.3f}")
    
    # Cuadrantes con alta densidad
    alta_densidad = densidades_validas > densidades_validas.quantile(0.95)
    n_alta_densidad = alta_densidad.sum()
    print(f"   • Cuadrantes > P95: {n_alta_densidad}")

# =============================================================================
# IDENTIFICAR CUADRANTES CON DENSIDAD EXTREMA
# =============================================================================

if n_cuadrantes_con_densidad > 0:
    # Top 5 densidades más altas
    top_densidades = df_final[area_valida].nlargest(5, 'density_naive_per_km2')
    
    print(f"\n🔝 Top 5 cuadrantes con mayor densidad:")
    for idx, row in top_densidades.iterrows():
        print(f"   • {row['cod_cuadrante']}: {row['density_naive_per_km2']:.2f} muestras/km² "
              f"({row['k_total']} muestras, {row['area_km2']:.4f} km²)")

# =============================================================================
# CUADRANTES SIN MUESTRAS (DENSIDAD = 0)
# =============================================================================

sin_muestras = df_final[(area_valida) & (df_final['k_total'] == 0)]
n_sin_muestras = len(sin_muestras)

print(f"\n🔍 Cuadrantes sin muestras: {n_sin_muestras:,}")

if n_sin_muestras > 0 and n_sin_muestras <= 20:
    print(f"   Listado completo:")
    for idx, row in sin_muestras.iterrows():
        print(f"   • {row['cod_cuadrante']}: 0 muestras ({row['area_km2']:.4f} km²)")
elif n_sin_muestras > 20:
    print(f"   (Lista muy larga, mostrando solo algunos ejemplos)")
    for idx, row in sin_muestras.head(10).iterrows():
        print(f"   • {row['cod_cuadrante']}: 0 muestras ({row['area_km2']:.4f} km²)")
    print(f"   ... y {n_sin_muestras - 10} más")

# =============================================================================
# VALIDACIÓN DE DENSIDADES
# =============================================================================

# Verificar que no hay densidades negativas
densidades_negativas = (df_final['density_naive_per_km2'] < 0).sum()
if densidades_negativas > 0:
    print(f"⚠️  ADVERTENCIA: {densidades_negativas} densidades negativas detectadas")

# Verificar consistencia entre densidades por m² y km²
if n_cuadrantes_con_densidad > 0:
    ratio_check = df_final[area_valida]['density_naive_per_km2'] / (df_final[area_valida]['density_naive_per_m2'] * 1e6)
    ratio_ok = np.allclose(ratio_check, 1.0, rtol=1e-6)
    if ratio_ok:
        print("✅ Consistencia entre densidades por m² y km² verificada")
    else:
        print("⚠️  ADVERTENCIA: Inconsistencia en conversión m²/km²")

print("✅ Cálculo de densidades naïve completado")

🎯 Calculando densidades naïve...
📊 Densidades calculadas para 30 cuadrantes

📈 Estadísticas de densidad naïve (muestras/km²):
   • Mínima: 64.6833
   • Cuartil 1: 637.160
   • Mediana: 1343.869
   • Cuartil 3: 3400.431
   • Máxima: 4957.96
   • Promedio: 1951.233
   • Desv. Estándar: 1609.788
   • Cuadrantes > P95: 2

🔝 Top 5 cuadrantes con mayor densidad:
   • MZ_012: 4957.96 muestras/km² (3181 muestras, 0.6416 km²)
   • MZ_030: 4463.00 muestras/km² (64 muestras, 0.0143 km²)
   • MZ_028: 4280.63 muestras/km² (2131 muestras, 0.4978 km²)
   • MZ_018: 4189.65 muestras/km² (1737 muestras, 0.4146 km²)
   • MZ_008: 3875.13 muestras/km² (792 muestras, 0.2044 km²)

🔍 Cuadrantes sin muestras: 0
✅ Consistencia entre densidades por m² y km² verificada
✅ Cálculo de densidades naïve completado


In [36]:
# ================================================================================================
# 7. FUNCIONES AUXILIARES PARA ANÁLISIS ESTADÍSTICO  
# ================================================================================================

print("📊 Definiendo funciones auxiliares...")

def calculate_credible_intervals_by_year(df, k_cols, area_col, priors_df, confidence=0.95):
    """
    Calcula intervalos de credibilidad para las tasas λᵢ por año usando posterior Gamma.
    
    Parameters:
    -----------
    df : DataFrame
        Datos con conteos y áreas
    k_cols : list
        Lista de columnas de conteos por año ['k_2021', 'k_2022', ...]  
    area_col : str
        Nombre de la columna de área
    priors_df : DataFrame
        Tabla de parámetros prior por año
    confidence : float
        Nivel de confianza (default 0.95)
        
    Returns:
    --------
    DataFrame con columnas de intervalos por año
    """
    try:
        from scipy.stats import gamma
        scipy_available = True
    except ImportError:
        print("⚠️  SciPy no disponible, saltando intervalos de credibilidad")
        return df.copy()
    
    df_result = df.copy()
    
    for k_col in k_cols:
        year = k_col.split('_')[1]  # Extraer año de 'k_2021' -> '2021'
        
        # Obtener parámetros del año
        year_params = priors_df[priors_df['anio'] == int(year)]
        if len(year_params) == 0:
            continue
            
        alpha_year = year_params.iloc[0]['alpha']
        beta_year = year_params.iloc[0]['beta']
        
        # Calcular intervalos
        lower_col = f'lower_95_{year}'
        upper_col = f'upper_95_{year}'
        
        df_result[lower_col] = np.nan
        df_result[upper_col] = np.nan
        
        # Máscara de valores válidos
        valid_mask = (
            df_result[k_col].notna() & 
            df_result[area_col].notna() & 
            (df_result[area_col] > 0) & 
            (df_result[k_col] >= 0)
        )
        
        if valid_mask.sum() > 0:
            # Parámetros posteriores
            k_valid = df_result.loc[valid_mask, k_col]
            A_valid = df_result.loc[valid_mask, area_col]
            
            alpha_post = alpha_year + k_valid
            beta_post = beta_year + A_valid
            
            # Cuantiles
            alpha_level = (1 - confidence) / 2
            lower_vals = gamma.ppf(alpha_level, alpha_post, scale=1/beta_post)
            upper_vals = gamma.ppf(1 - alpha_level, alpha_post, scale=1/beta_post)
            
            df_result.loc[valid_mask, lower_col] = lower_vals
            df_result.loc[valid_mask, upper_col] = upper_vals
    
    return df_result

def summarize_idba_results_by_year(df, idba_cols):
    """
    Genera resumen estadístico de resultados IDBA por año.
    
    Parameters:
    -----------
    df : DataFrame
        Datos con columnas IDBA
    idba_cols : list  
        Lista de columnas IDBA ['idba_2021', 'idba_2022', ...]
        
    Returns:
    --------
    DataFrame con estadísticas por año
    """
    summaries = []
    
    for col in idba_cols:
        if col not in df.columns:
            continue
            
        year = col.split('_')[1] if '_' in col else col
        values = df[col].dropna()
        
        if len(values) > 0:
            summaries.append({
                'año': year,
                'n_válidos': len(values),
                'mínimo': values.min(),
                'percentil_25': values.quantile(0.25),
                'mediana': values.median(), 
                'percentil_75': values.quantile(0.75),
                'máximo': values.max(),
                'promedio': values.mean(),
                'desv_estándar': values.std()
            })
    
    return pd.DataFrame(summaries)

def validate_idba_coherence(df, naive_cols, idba_cols):
    """
    Valida coherencia entre estimaciones naïve e IDBA.
    
    Parameters:
    -----------
    df : DataFrame
        Datos con columnas naïve e IDBA
    naive_cols : list
        Columnas de densidades naïve
    idba_cols : list  
        Columnas de densidades IDBA
        
    Returns:
    --------  
    dict con métricas de validación
    """
    validations = {}
    
    for naive_col, idba_col in zip(naive_cols, idba_cols):
        if naive_col not in df.columns or idba_col not in df.columns:
            continue
            
        # Datos válidos para ambos
        both_valid = df[naive_col].notna() & df[idba_col].notna()
        
        if both_valid.sum() < 2:
            continue
            
        naive_vals = df.loc[both_valid, naive_col]
        idba_vals = df.loc[both_valid, idba_col]
        
        # Correlación
        correlation = naive_vals.corr(idba_vals)
        
        # Diferencia relativa promedio
        rel_diff = abs(naive_vals - idba_vals) / (naive_vals + 1e-10)
        mean_rel_diff = rel_diff.mean()
        
        # Ratio promedio IDBA/Naïve
        ratio = idba_vals / (naive_vals + 1e-10)
        mean_ratio = ratio.mean()
        
        year = idba_col.split('_')[1] if '_' in idba_col else 'total'
        
        validations[year] = {
            'correlación': correlation,
            'dif_relativa_media': mean_rel_diff,
            'ratio_idba_naive': mean_ratio,
            'n_comparaciones': both_valid.sum()
        }
    
    return validations

print("✅ Funciones auxiliares definidas correctamente")

📊 Definiendo funciones auxiliares...
✅ Funciones auxiliares definidas correctamente


In [37]:
# ================================================================================================
# 8. APLICAR IDBA ROBUSTO POR AÑO  (VERSIÓN VECTORIZADA Y LIGERA)
# ================================================================================================
print(">> IDBA Poisson–Gamma robusto por año (vectorizado) ...")

# -----------------------------------------------------------------
# Configuración del prior robusto
# -----------------------------------------------------------------
ANIOS = [2021, 2022, 2023, 2024]
E0_MODE = 'p20'    # 'p20' recomendado; alternativo: 'fixed'
E0_FIXED = 0.05    # km², solo si E0_MODE == 'fixed'

# Asegurar dtypes numéricos
df_final['area_km2'] = pd.to_numeric(df_final['area_km2'], errors='coerce')
for _y in ANIOS + ['total']:
    kcol = f'k_{_y}' if _y != 'total' else 'k_total'
    if kcol in df_final.columns:
        df_final[kcol] = pd.to_numeric(df_final[kcol], errors='coerce')

# Filtrado para prior (excluye FUERA y áreas inválidas)
mask_prior = (df_final['cod_cuadrante'] != 'FUERA') & (df_final['area_km2'] > 0)
df_prior = df_final.loc[mask_prior, ['cod_cuadrante', 'area_km2'] + [f'k_{y}' for y in ANIOS if f'k_{y}' in df_final.columns]].copy()

# Si no hay suficientes cuadrantes válidos, salir con NaN
if df_prior.empty:
    for y in ANIOS:
        df_final[f'idba_{y}'] = np.nan
        df_final[f'weight_prior_{y}'] = np.nan
    df_final['idba_total'] = np.nan
    print("⚠️ No hay cuadrantes válidos para calcular priors/IDBA.")
else:
    A = df_prior['area_km2'].to_numpy()                      # (n,)
    if E0_MODE == 'p20':
        E0_value = float(np.nanpercentile(A, 20))
        E0_desc = f"P20 áreas = {E0_value:.4f} km²"
    elif E0_MODE == 'fixed':
        E0_value = float(E0_FIXED)
        E0_desc = f"Fijo = {E0_value:.4f} km²"
    else:
        raise ValueError("E0_MODE debe ser 'p20' o 'fixed'")

    # Tabla de priors por año (vectorizada)
    priors_rows = []
    for y in ANIOS:
        kcol = f'k_{y}'
        if kcol not in df_prior.columns:
            continue
        K = df_prior[kcol].to_numpy()                        # (n,)
        # tasas r = K/A (km^-2), cuidando división
        R = np.divide(K, A, out=np.full_like(K, np.nan, dtype=float), where=A > 0)

        # mu0 = mediana robusta de R; fallback a promedio total si todo NaN/cero
        if np.isfinite(R).any():
            mu0 = float(np.nanmedian(R))
        else:
            totK = float(np.nansum(K))
            totA = float(np.nansum(A))
            mu0 = (totK / totA) if totA > 0 else 0.0

        alpha = mu0 * E0_value
        beta = E0_value

        priors_rows.append({
            'anio': y, 'mu0': mu0, 'E0': E0_value, 'alpha': alpha, 'beta': beta,
            'total_k': int(np.nansum(K)), 'total_area': float(np.nansum(A)),
            'n_quadrants_used': int(np.sum(np.isfinite(A))), 'E0_mode': E0_MODE, 'E0_value': E0_value
        })

    df_priors = pd.DataFrame(priors_rows).sort_values('anio')
    print(f"   • E0: {E0_desc}")
    print("   • Priors calculados por año:\n", df_priors[['anio','mu0','alpha','beta']].to_string(index=False, float_format='%.6f'))

    # Aplicar IDBA por año (asignación por vector, sin .loc dentro de bucles)
    idba_mask = (df_final['area_km2'] > 0)
    A_all = df_final['area_km2'].to_numpy()
    for y in ANIOS:
        kcol = f'k_{y}'
        idcol = f'idba_{y}'
        if kcol not in df_final.columns or df_priors[df_priors['anio']==y].empty:
            df_final[idcol] = np.nan
            df_final[f'weight_prior_{y}'] = np.nan
            continue

        pr = df_priors.loc[df_priors['anio']==y].iloc[0]
        aY = float(pr['alpha']); bY = float(pr['beta'])

        K_all = pd.to_numeric(df_final[kcol], errors='coerce').to_numpy()
        # ρ̂ = (α + K) / (β + A)  (km^-2). Usamos np.divide para velocidad/estabilidad numérica.
        num = aY + K_all
        den = bY + A_all
        idba = np.divide(num, den, out=np.full_like(den, np.nan, dtype=float), where=(den > 0) & idba_mask.to_numpy())
        df_final[idcol] = idba

        # Peso del prior (shrinkage): w = E0 / (E0 + A)
        w = np.divide(bY, den, out=np.full_like(den, np.nan, dtype=float), where=(den > 0) & idba_mask.to_numpy())
        df_final[f'weight_prior_{y}'] = w

    # IDBA total (4 años) – opcional si ya lo usabas
    if 'k_total' in df_final.columns:
        # prior total con mismo E0_value y mu0_total=mediana(k_total/A)
        Ktot = df_prior['k_total'].to_numpy() if 'k_total' in df_prior.columns else None
        if Ktot is None:
            # si df_prior no trae k_total, lo calculamos ad-hoc
            Ktot = (df_prior[[f'k_{y}' for y in ANIOS if f'k_{y}' in df_prior.columns]]
                    .to_numpy(dtype=float)).sum(axis=1)
        Rtot = np.divide(Ktot, A, out=np.full_like(Ktot, np.nan, dtype=float), where=A > 0)
        mu0_tot = float(np.nanmedian(Rtot)) if np.isfinite(Rtot).any() else (
            float(np.nansum(Ktot))/float(np.nansum(A)) if np.nansum(A)>0 else 0.0
        )
        alpha_tot = mu0_tot * E0_value
        beta_tot  = E0_value

        numT = alpha_tot + pd.to_numeric(df_final['k_total'], errors='coerce').to_numpy()
        denT = beta_tot + A_all
        idbaT = np.divide(numT, denT, out=np.full_like(denT, np.nan, dtype=float), where=(denT > 0) & idba_mask.to_numpy())
        df_final['idba_total'] = idbaT

        print("   • IDBA total aplicado.")
    else:
        df_final['idba_total'] = np.nan

    # Resumen rápido del shrinkage (min/med/max) sin impresiones pesadas
    wstats = []
    for y in ANIOS:
        wcol = f'weight_prior_{y}'
        if wcol in df_final.columns:
            w = df_final.loc[idba_mask, wcol].to_numpy()
            w = w[np.isfinite(w)]
            if w.size:
                wstats.append((y, np.min(w), np.median(w), np.max(w), np.mean(w)))
    if wstats:
        print("   • Shrinkage (peso del prior) min/med/max/avg por año:")
        for (y, mn, md, mx, av) in wstats:
            print(f"     - {y}: {mn:.3f}/{md:.3f}/{mx:.3f}/{av:.3f}")

print("✅ IDBA robusto por año (vectorizado) completado.")

>> IDBA Poisson–Gamma robusto por año (vectorizado) ...
   • E0: P20 áreas = 0.1980 km²
   • Priors calculados por año:
  anio        mu0      alpha     beta
 2021   3.752861   0.743049 0.197995
 2022   0.000000   0.000000 0.197995
 2023 434.611115  86.050912 0.197995
 2024 648.139202 128.328447 0.197995
   • IDBA total aplicado.
   • Shrinkage (peso del prior) min/med/max/avg por año:
     - 2021: 0.133/0.288/0.932/0.374
     - 2022: 0.133/0.288/0.932/0.374
     - 2023: 0.133/0.288/0.932/0.374
     - 2024: 0.133/0.288/0.932/0.374
✅ IDBA robusto por año (vectorizado) completado.


In [ ]:
# ================================================================================================
# 8.1. ANALISIS DE RESULTADOS IDBA
# ================================================================================================

print("Analizando resultados IDBA...")

# =============================================================================
# ESTADISTICAS BASICAS IDBA
# =============================================================================

# Resumen por ano
for y in ANIOS:
    idba_col = f'idba_{y}'
    if idba_col in df_final.columns:
        vals = df_final[idba_col].dropna()
        if len(vals) > 0:
            print(f"IDBA {y}: {len(vals):,} valores, "
                  f"min={vals.min():.4f}, med={vals.median():.4f}, max={vals.max():.4f}")

# IDBA total
if 'idba_total' in df_final.columns:
    vals_total = df_final['idba_total'].dropna()
    if len(vals_total) > 0:
        print(f"IDBA total: {len(vals_total):,} valores, "
              f"min={vals_total.min():.4f}, med={vals_total.median():.4f}, max={vals_total.max():.4f}")

# =============================================================================
# ANALISIS DE SHRINKAGE
# =============================================================================

print("\nAnalisis de shrinkage:")
for y in ANIOS:
    wcol = f'weight_prior_{y}'
    if wcol in df_final.columns:
        w = df_final[wcol].dropna()
        if len(w) > 0:
            print(f"Peso prior {y}: min={w.min():.3f}, med={w.median():.3f}, max={w.max():.3f}")
            
# =============================================================================
# COMPARACION NAIVE vs IDBA
# =============================================================================

if 'density_naive_per_km2' in df_final.columns and 'idba_total' in df_final.columns:
    both_valid = df_final['density_naive_per_km2'].notna() & df_final['idba_total'].notna()
    if both_valid.sum() > 0:
        corr = df_final.loc[both_valid, 'density_naive_per_km2'].corr(df_final.loc[both_valid, 'idba_total'])
        print(f"\nCorrelacion naive vs IDBA total: {corr:.4f}")

print("\nAnalisis IDBA completado.")
    except AssertionError as e:
        print(f"   ❌ ERROR IDBA total: {e}")
        raise

# 4) Verificar que 'FUERA' tiene NaN en IDBA (como debe ser)
print("4️⃣ Validando cuadrante FUERA...")
fuera_mask = df_final['cod_cuadrante'] == 'FUERA'
if fuera_mask.any():
    fuera_row = df_final[fuera_mask].iloc[0]
    
    # Verificar que FUERA tiene NaN en todos los IDBA
    idba_cols_check = [f'idba_{y}' for y in ANIOS if f'idba_{y}' in df_final.columns] + (['idba_total'] if 'idba_total' in df_final.columns else [])
    
    for idba_col in idba_cols_check:
        fuera_idba_value = fuera_row[idba_col]
        if not pd.isna(fuera_idba_value):
            print(f"   ⚠️  ADVERTENCIA: 'FUERA' tiene valor IDBA no-NaN en {idba_col}: {fuera_idba_value}")
        else:
            print(f"   ✅ 'FUERA' correctamente NaN en {idba_col}")
            
    # Verificar que FUERA tiene área NaN o inválida
    fuera_area = fuera_row['area_km2']
    if pd.isna(fuera_area) or fuera_area <= 0:
        print(f"   ✅ 'FUERA' correctamente sin área válida: {fuera_area}")
    else:
        print(f"   ⚠️  ADVERTENCIA: 'FUERA' tiene área válida: {fuera_area} km²")
else:
    print("   ℹ️  No se encontró cuadrante 'FUERA' en los datos")

# 5) Verificar que los valores IDBA son no-negativos donde están definidos
print("5️⃣ Validando no-negatividad de IDBA...")
for y in ANIOS + ['total']:
    idba_col = f'idba_{y}' if y != 'total' else 'idba_total'
    if idba_col in df_final.columns:
        valores_validos = df_final[idba_col].dropna()
        negativos = (valores_validos < 0).sum()
        
        if negativos == 0:
            print(f"   ✅ {idba_col}: todos los valores son no-negativos ({len(valores_validos):,} valores)")
        else:
            print(f"   ❌ ERROR {idba_col}: {negativos} valores negativos detectados")
            raise ValueError(f"Valores IDBA negativos en {idba_col}")

# =============================================================================
# RESUMEN QA
# =============================================================================

print("\n" + "="*50)
print("✅ QA MÍNIMO COMPLETADO EXITOSAMENTE")
print("="*50)
print(f"📊 Total muestras: {conteo_total_df:,}")
print(f"📊 Cuadrantes válidos: {cuadrantes_validos:,}")
print(f"📊 Años procesados: {ANIOS}")
print(f"🔍 Todas las validaciones pasaron correctamente")
print("="*50)

print("QA mínimo OK")

IndentationError: unexpected indent (4284850640.py, line 50)

: 

In [47]:
# ================================================================================================
# 8. UTILS - EB IDBA (vectorizado, sin dependencias rotas)
# ================================================================================================

import numpy as np
import pandas as pd

ANIOS = [2021, 2022, 2023, 2024]

def _num(s):
    """Convierte a numerico seguro"""
    return pd.to_numeric(s, errors='coerce')

def compute_priors_eb(df_final, anos=ANIOS, e0_mode='p20', e0_fixed=0.05):
    """
    Priors EB robustos por ano:
      - mu0 = mediana de tasas (k_y / area_km2)
      - E0 = p20 de areas_km2 (o valor fijo)
      - alpha = mu0 * E0, beta = E0   (Gamma shape/rate)
    Excluye 'FUERA' y areas <= 0.
    Retorna DataFrame: anio, mu0, E0, alpha, beta, total_k, total_area, n_quadrants_used, E0_mode, E0_value
    """
    df = df_final.copy()
    df['area_km2'] = _num(df['area_km2'])
    for y in anos:
        kcol = f'k_{y}'
        if kcol in df.columns:
            df[kcol] = _num(df[kcol])

    # filtro prior
    m = (df['cod_cuadrante'] != 'FUERA') & (df['area_km2'] > 0)
    dfp = df.loc[m, ['cod_cuadrante','area_km2'] + [f'k_{y}' for y in anos if f'k_{y}' in df.columns]].copy()
    if dfp.empty:
        return pd.DataFrame(columns=['anio','mu0','E0','alpha','beta','total_k','total_area','n_quadrants_used','E0_mode','E0_value'])

    A = dfp['area_km2'].to_numpy(dtype=float)
    if e0_mode == 'p20':
        E0_value = float(np.nanpercentile(A, 20))
    elif e0_mode == 'fixed':
        E0_value = float(e0_fixed)
    else:
        raise ValueError("e0_mode debe ser 'p20' o 'fixed'")

    rows = []
    for y in anos:
        kcol = f'k_{y}'
        if kcol not in dfp.columns:
            continue
        K = dfp[kcol].to_numpy(dtype=float)
        R = np.divide(K, A, out=np.full_like(A, np.nan, dtype=float), where=A > 0)
        if np.isfinite(R).any():
            mu0 = float(np.nanmedian(R))
        else:
            totK = float(np.nansum(K)); totA = float(np.nansum(A))
            mu0 = (totK / totA) if totA > 0 else 0.0
        rows.append({
            'anio': y,
            'mu0': mu0,
            'E0': E0_value,
            'alpha': mu0 * E0_value,
            'beta': E0_value,
            'total_k': int(np.nansum(K)),
            'total_area': float(np.nansum(A)),
            'n_quadrants_used': int(np.sum(np.isfinite(A))),
            'E0_mode': e0_mode,
            'E0_value': E0_value if e0_mode == 'fixed' else np.nan
        })

    # prior total 4Y opcional
    if 'k_total' in df.columns:
        Ktot = df.loc[m, 'k_total'].to_numpy(dtype=float)
        Rtot = np.divide(Ktot, A, out=np.full_like(A, np.nan, dtype=float), where=A > 0)
        if np.isfinite(Rtot).any():
            mu0_tot = float(np.nanmedian(Rtot))
        else:
            mu0_tot = (float(np.nansum(Ktot))/float(np.nansum(A))) if np.nansum(A) > 0 else 0.0
        rows.append({
            'anio': 'TOTAL',
            'mu0': mu0_tot,
            'E0': E0_value,
            'alpha': mu0_tot * E0_value,
            'beta': E0_value,
            'total_k': int(np.nansum(Ktot)),
            'total_area': float(np.nansum(A)),
            'n_quadrants_used': int(np.sum(np.isfinite(A))),
            'E0_mode': e0_mode,
            'E0_value': E0_value if e0_mode == 'fixed' else np.nan
        })

    df_result = pd.DataFrame(rows)
    # Sort with years first, then TOTAL at end
    if not df_result.empty:
        year_rows = df_result[df_result['anio'] != 'TOTAL'].copy()
        total_rows = df_result[df_result['anio'] == 'TOTAL'].copy()
        if not year_rows.empty:
            year_rows = year_rows.sort_values('anio')
        df_result = pd.concat([year_rows, total_rows], ignore_index=True)
    return df_result

def apply_idba_from_priors(df_final, priors, anos=ANIOS):
    """
    IDBA por ano: idba_y = (alpha_y + k_y) / (beta_y + area_km2).
    Crea weight_prior_y = beta_y / (beta_y + area_km2) para diagnosticar shrinkage.
    Devuelve df_final modificado.
    """
    df = df_final
    df['area_km2'] = _num(df['area_km2'])
    idba_mask = df['area_km2'] > 0
    A_all = df['area_km2'].to_numpy(dtype=float)

    for y in anos:
        kcol = f'k_{y}'; idcol = f'idba_{y}'; wcol = f'weight_prior_{y}'
        if (kcol not in df.columns) or priors[priors['anio']==y].empty:
            df[idcol] = np.nan; df[wcol] = np.nan
            continue
        pr = priors.loc[priors['anio']==y].iloc[0]
        aY = float(pr['alpha']); bY = float(pr['beta'])
        K_all = _num(df[kcol]).to_numpy(dtype=float)
        den = bY + A_all
        df[idcol] = np.divide(aY + K_all, den, out=np.full_like(den, np.nan, dtype=float), where=(den > 0) & idba_mask.to_numpy())
        df[wcol]  = np.divide(bY, den, out=np.full_like(den, np.nan, dtype=float), where=(den > 0) & idba_mask.to_numpy())

    # total 4Y opcional
    if ('k_total' in df.columns) and (not priors[priors['anio']=='TOTAL'].empty):
        pr = priors.loc[priors['anio']=='TOTAL'].iloc[0]
        aT = float(pr['alpha']); bT = float(pr['beta'])
        Ktot = _num(df['k_total']).to_numpy(dtype=float)
        denT = bT + A_all
        df['idba_total'] = np.divide(aT + Ktot, denT, out=np.full_like(denT, np.nan, dtype=float), where=(denT > 0) & idba_mask.to_numpy())
    else:
        df['idba_total'] = np.nan

    return df

def compute_credible_intervals_gamma(df_final, priors, anos=ANIOS, conf=0.95):
    """
    Intervalos de credibilidad para rho posterior (Gamma(alpha+k, rate=beta+A)).
    Requiere SciPy. Si no esta disponible, devuelve None y el caller debe manejarlo.
    """
    try:
        from scipy.stats import gamma as gamma_dist
    except Exception:
        return None  # sin SciPy -> caller imprime aviso y omite CIs

    lower_q = (1.0 - conf) / 2.0
    upper_q = 1.0 - lower_q

    res = {}
    A = _num(df_final['area_km2']).to_numpy(dtype=float)
    ok = (A > 0) & (df_final['cod_cuadrante'] != 'FUERA')

    for y in anos:
        kcol = f'k_{y}'
        if (kcol not in df_final.columns) or priors[priors['anio']==y].empty:
            res[y] = (np.full(df_final.shape[0], np.nan), np.full(df_final.shape[0], np.nan))
            continue
        pr = priors.loc[priors['anio']==y].iloc[0]
        aY = float(pr['alpha']); bY = float(pr['beta'])
        K = _num(df_final[kcol]).to_numpy(dtype=float)
        shape = aY + K
        rate  = bY + A
        scale = np.divide(1.0, rate, out=np.full_like(rate, np.nan, dtype=float), where=rate>0)

        lo = np.full(df_final.shape[0], np.nan, dtype=float)
        hi = np.full(df_final.shape[0], np.nan, dtype=float)
        idx = np.where(ok & np.isfinite(shape) & np.isfinite(scale))[0]
        if idx.size:
            lo[idx] = gamma_dist.ppf(lower_q, a=shape[idx], scale=scale[idx])
            hi[idx] = gamma_dist.ppf(upper_q, a=shape[idx], scale=scale[idx])
        res[y] = (lo, hi)

    return res

print("Utils EB IDBA definidas correctamente.")

Utils EB IDBA definidas correctamente.


In [48]:
# ================================================================================================
# 9. PRIORS EB por ano (mu0 mediana tasas, E0 p20 o fijo)
# ================================================================================================

E0_MODE = 'p20'    # 'p20' recomendado; alternativo: 'fixed'
E0_FIXED = 0.05    # km^2 si E0_MODE == 'fixed'

print("Calculando priors EB robustos por ano...")

priors = compute_priors_eb(df_final, anos=ANIOS, e0_mode=E0_MODE, e0_fixed=E0_FIXED)

print("Priors calculados:")
display(priors)

# Resumen de shrinkage potencial (aprox, constante por ano): w = E0/(E0 + A)
valid_areas = df_final.loc[(df_final['cod_cuadrante']!='FUERA') & (df_final['area_km2']>0), 'area_km2'].to_numpy(dtype=float)
if not priors.empty and valid_areas.size:
    E0v = float(priors.iloc[0]['E0'])
    w = E0v / (E0v + valid_areas)
    print("Shrinkage esperado aprox (min/med/max/avg):",
          f"{np.min(w):.3f}/{np.median(w):.3f}/{np.max(w):.3f}/{np.mean(w):.3f}")

print("Priors EB completados.")

Calculando priors EB robustos por ano...
Priors calculados:


,anio,mu0,E0,alpha,beta,total_k,total_area,n_quadrants_used,E0_mode,E0_value
0,2021,3.752861,0.197995,0.743049,0.197995,7916,13.663576,30,p20,NaN
1,2022,0.000000,0.197995,0.000000,0.197995,0,13.663576,30,p20,NaN
2,2023,434.611115,0.197995,86.050912,0.197995,8172,13.663576,30,p20,NaN
3,2024,648.139202,0.197995,128.328447,0.197995,9781,13.663576,30,p20,NaN
4,TOTAL,1343.868637,0.197995,266.079531,0.197995,25869,13.663576,30,p20,NaN


Shrinkage esperado aprox (min/med/max/avg): 0.133/0.288/0.932/0.374
Priors EB completados.


In [49]:
# ================================================================================================
# 10. Aplicar IDBA (vectorizado) y CIs (opcional)
# ================================================================================================

print("Aplicando IDBA por ano (vectorizado)...")

# Aplicar IDBA usando los priors calculados
df_final = apply_idba_from_priors(df_final, priors, anos=ANIOS)

# (Opcional) Intervalos de credibilidad si SciPy esta disponible
cis = compute_credible_intervals_gamma(df_final, priors, anos=ANIOS, conf=0.95)
if cis is None:
    print("Aviso: SciPy no disponible. Se omiten intervalos de credibilidad.")
else:
    print("Calculando intervalos de credibilidad 95%...")
    for y in ANIOS:
        lo, hi = cis.get(y, (None, None))
        if lo is not None:
            df_final[f'idba_{y}_lo95'] = lo
            df_final[f'idba_{y}_hi95'] = hi
    print("Intervalos de credibilidad calculados.")

# Vista rapida
cols_show = ['cod_cuadrante','area_km2'] + [f'k_{y}' for y in ANIOS if f'k_{y}' in df_final.columns] + [f'idba_{y}' for y in ANIOS if f'idba_{y}' in df_final.columns]
print("Vista rapida de resultados IDBA:")
display(df_final[cols_show].head(12))

print("IDBA aplicado exitosamente.")

Aplicando IDBA por ano (vectorizado)...
Calculando intervalos de credibilidad 95%...
Intervalos de credibilidad calculados.
Vista rapida de resultados IDBA:
Calculando intervalos de credibilidad 95%...
Intervalos de credibilidad calculados.
Vista rapida de resultados IDBA:


,cod_cuadrante,area_km2,k_2021,k_2022,k_2023,k_2024,idba_2021,idba_2022,idba_2023,idba_2024
0,FUERA,NaN,222,21,395,470,NaN,NaN,NaN,NaN
1,MZ_001,0.101063,0,0,62,0,2.484630,0.0,495.057481,429.108857
2,MZ_002,0.756543,0,0,563,0,0.778438,0.0,679.963306,134.440355
3,MZ_003,0.831687,1,0,588,0,1.692803,0.0,654.620573,124.629223
4,MZ_004,0.206231,0,0,0,167,1.838199,0.0,212.878020,730.601610
5,MZ_005,0.354554,131,0,279,88,238.427831,0.0,660.667095,391.510011
6,MZ_006,0.532286,0,0,0,39,1.017483,0.0,117.832594,229.128832
7,MZ_007,0.639571,0,0,226,988,0.887152,0.0,372.568546,1332.823748
8,MZ_008,0.204380,362,0,199,231,901.504210,0.0,708.420460,893.018099
9,MZ_009,0.594337,0,0,936,1055,0.937799,0.0,1289.926694,1493.474477


IDBA aplicado exitosamente.


In [50]:
# ================================================================================================
# 11. Exportar Priors y QA Minimo
# ================================================================================================

print("Exportando priors y ejecutando QA minimo...")

# Export priors
priors.to_csv("idba_priors_MANIZALES_2021-2024.csv", index=False, sep=';', encoding='utf-8-sig')
print("Priors exportados a: idba_priors_MANIZALES_2021-2024.csv")

# QA minimo
print("\nEjecutando QA minimo...")

# 1) Conservacion de masa
conteo_original = len(df_m[df_m['ANIO'].isin(ANIOS)])
conteo_final = df_final['k_total'].fillna(0).sum()
assert conteo_final == conteo_original, f"Conteo total no coincide: final={conteo_final}, original={conteo_original}"
print(f"   Conservacion de masa OK: {conteo_final:,} muestras procesadas")

# 2) IDBA NaN solo en FUERA o area<=0
ok = (df_final['cod_cuadrante']!='FUERA') & (df_final['area_km2']>0)
for y in ANIOS:
    col = f'idba_{y}'
    if col in df_final.columns:
        nan_count = df_final.loc[ok, col].isna().sum()
        assert nan_count == 0, f"IDBA {y} contiene {nan_count} NaN inesperados."
        print(f"   IDBA {y}: sin NaN inesperados ({ok.sum():,} cuadrantes validos)")

# 3) IDBA total tambien
if 'idba_total' in df_final.columns:
    nan_total = df_final.loc[ok, 'idba_total'].isna().sum()
    assert nan_total == 0, f"IDBA total contiene {nan_total} NaN inesperados."
    print(f"   IDBA total: sin NaN inesperados ({ok.sum():,} cuadrantes validos)")

print("\nQA minimo OK.")

Exportando priors y ejecutando QA minimo...
Priors exportados a: idba_priors_MANIZALES_2021-2024.csv

Ejecutando QA minimo...
   Conservacion de masa OK: 26,977 muestras procesadas
   IDBA 2021: sin NaN inesperados (30 cuadrantes validos)
   IDBA 2022: sin NaN inesperados (30 cuadrantes validos)
   IDBA 2023: sin NaN inesperados (30 cuadrantes validos)
   IDBA 2024: sin NaN inesperados (30 cuadrantes validos)
   IDBA total: sin NaN inesperados (30 cuadrantes validos)

QA minimo OK.


In [45]:
# ================================================================================================
# 10. PREPARAR Y EXPORTAR RESULTADOS FINALES
# ================================================================================================

print("💾 Preparando exportación de resultados...")

# =============================================================================
# SELECCIONAR Y ORDENAR COLUMNAS PARA EXPORTACIÓN
# =============================================================================

# Definir columnas para la tabla final
columnas_exportacion = [
    # Identificación
    'cod_cuadrante',
    
    # Datos básicos
    'k_total', 'k_2021', 'k_2022', 'k_2023', 'k_2024',
    'dias_any_total', 'dias_ge2_total',
    'area_m2', 'area_km2',
    
    # Densidades naïve
    'density_naive_per_m2', 'density_naive_per_km2',
    
    # Resultados IDBA por año
    'idba_total', 'idba_2021', 'idba_2022', 'idba_2023', 'idba_2024',
    
    # Análisis de shrinkage
    'weight_prior_2021', 'weight_prior_2022', 'weight_prior_2023', 'weight_prior_2024'
]

# Filtrar solo las columnas que existen
columnas_disponibles = [col for col in columnas_exportacion if col in df_final.columns]
print(f"📋 Columnas para exportar: {len(columnas_disponibles)}/{len(columnas_exportacion)}")

# Crear dataframe para exportación
df_export = df_final[columnas_disponibles].copy()

# =============================================================================
# ORDENAR RESULTADOS
# =============================================================================

# Ordenar por IDBA total descendente, luego por código de cuadrante
if 'idba_total' in df_export.columns:
    df_export = df_export.sort_values(
        ['idba_total', 'cod_cuadrante'], 
        ascending=[False, True],
        na_position='last'
    )
else:
    df_export = df_export.sort_values('cod_cuadrante')

print(f"📊 Registros ordenados: {len(df_export):,}")

# =============================================================================
# AGREGAR RANKINGS Y PERCENTILES
# =============================================================================

# Ranking por IDBA total
if 'idba_total' in df_export.columns:
    mask_valido = df_export['idba_total'].notna()
    df_export.loc[mask_valido, 'rank_idba_total'] = (
        df_export[mask_valido]['idba_total'].rank(method='dense', ascending=False)
    )
    
    # Percentiles
    densidades = df_export['idba_total'].dropna()
    if len(densidades) > 0:
        df_export.loc[mask_valido, 'percentile_idba_total'] = (
            df_export[mask_valido]['idba_total'].rank(pct=True) * 100
        )

# =============================================================================
# ESTADÍSTICAS FINALES PARA METADATOS
# =============================================================================

metadata = {
    'fecha_procesamiento': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_cuadrantes': len(df_export),
    'cuadrantes_con_muestras': (df_export['k_total'] > 0).sum() if 'k_total' in df_export.columns else 'N/A',
    'cuadrantes_con_area': df_export['area_km2'].notna().sum() if 'area_km2' in df_export.columns else 'N/A',
    'total_muestras': df_export['k_total'].sum() if 'k_total' in df_export.columns else 'N/A',
    'area_total_km2': df_export['area_km2'].sum() if 'area_km2' in df_export.columns else 'N/A',
    'periodo_datos': '2021-2024',
    'ciudad': 'MANIZALES',
    'metodologia': 'IDBA Poisson-Gamma (EB robusto por año)',
    'E0_mode': E0_MODE,
    'E0_value': E0_value
}

# Agregar estadísticas de IDBA por año
for year in ANIOS:
    idba_col = f'idba_{year}'
    if idba_col in df_export.columns:
        valores_year = df_export[idba_col].dropna()
        if len(valores_year) > 0:
            metadata[f'idba_{year}_media'] = float(valores_year.mean())
            metadata[f'idba_{year}_mediana'] = float(valores_year.median())

# Agregar parámetros μ0 por año (trazabilidad)
for _, row in df_priors.iterrows():
    year = row['anio']
    if year != 'TOTAL':
        metadata[f'mu0_{year}'] = float(row['mu0'])
    else:
        metadata['mu0_total'] = float(row['mu0'])

print("\n📈 Estadísticas finales:")
for key, value in metadata.items():
    if isinstance(value, float):
        print(f"   • {key}: {value:.4f}")
    else:
        print(f"   • {key}: {value}")

# =============================================================================
# EXPORTAR ARCHIVOS
# =============================================================================

print(f"\n💾 Exportando archivos...")

# Definir rutas de salida
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
archivo_principal = ARCHIVO_SALIDA_INDICADORES.name.replace('.csv', f'_{timestamp}.csv')
archivo_priors = ARCHIVO_SALIDA_PRIORS.name.replace('.csv', f'_{timestamp}.csv')  
archivo_metadatos = f"metadatos_muestras_manizales_{timestamp}.json"

try:
    # Exportar tabla principal
    df_export.to_csv(archivo_principal, index=False, encoding='utf-8-sig', sep=';')
    print(f"   ✅ Tabla principal: {archivo_principal}")
    
    # Exportar tabla de priors
    df_priors.to_csv(archivo_priors, index=False, encoding='utf-8-sig', sep=';')
    print(f"   ✅ Priors por año: {archivo_priors}")
    
    # Exportar metadatos
    import json
    with open(archivo_metadatos, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
    print(f"   ✅ Metadatos: {archivo_metadatos}")
    
    # Verificar archivos creados
    import os
    size_principal = os.path.getsize(archivo_principal)
    size_priors = os.path.getsize(archivo_priors)
    size_metadatos = os.path.getsize(archivo_metadatos)
    
    print(f"\n📁 Archivos creados:")
    print(f"   • {archivo_principal}: {size_principal:,} bytes")
    print(f"   • {archivo_priors}: {size_priors:,} bytes")
    print(f"   • {archivo_metadatos}: {size_metadatos:,} bytes")
    
except Exception as e:
    print(f"❌ Error en exportación: {str(e)}")
    
# =============================================================================
# MOSTRAR MUESTRA DE RESULTADOS
# =============================================================================

print(f"\n🔍 Muestra de resultados (top 10 por IDBA total):")

columnas_muestra = ['cod_cuadrante', 'k_total', 'area_km2', 'idba_total', 'idba_2021', 'idba_2022', 'idba_2023', 'idba_2024']
columnas_muestra_disponibles = [col for col in columnas_muestra if col in df_export.columns]

muestra = df_export[columnas_muestra_disponibles].head(10)
print(muestra.to_string(index=False, float_format='%.4f'))

# =============================================================================
# MOSTRAR TABLA DE PRIORS
# =============================================================================

print(f"\n📋 Parámetros EB por año exportados:")
print(df_priors[['anio', 'mu0', 'alpha', 'beta', 'total_k', 'n_quadrants_used']].to_string(index=False, float_format='%.4f'))

# =============================================================================
# RESUMEN FINAL
# =============================================================================

print(f"\n" + "="*80)
print("🎯 PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*80)
print(f"📊 Cuadrantes procesados: {len(df_export):,}")
print(f"📊 Muestras analizadas: {metadata['total_muestras']:,}")
print(f"📊 Área cubierta: {metadata['area_total_km2']:.2f} km²")
print(f"🧠 Metodología: {metadata['metodologia']}")
print(f"📅 Período: {metadata['periodo_datos']}")
print(f"🏙️ Ciudad: {metadata['ciudad']}")
print(f"📏 Prior baseline: E0 = {E0_value:.4f} km² ({E0_MODE})")
print(f"💾 Archivos exportados: {archivo_principal}, {archivo_priors}, {archivo_metadatos}")
print("="*80)

💾 Preparando exportación de resultados...
📋 Columnas para exportar: 21/21
📊 Registros ordenados: 31

📈 Estadísticas finales:
   • fecha_procesamiento: 2025-10-14 15:27:06
   • total_cuadrantes: 31
   • cuadrantes_con_muestras: 31
   • cuadrantes_con_area: 30
   • total_muestras: 26977
   • area_total_km2: 13.6636
   • periodo_datos: 2021-2024
   • ciudad: MANIZALES
   • metodologia: IDBA Poisson-Gamma (EB robusto por año)
   • E0_mode: p20
   • E0_value: 0.1980
   • idba_2021_media: 387.2943
   • idba_2021_mediana: 3.7951
   • idba_2022_media: 0.0000
   • idba_2022_mediana: 0.0000
   • idba_2023_media: 543.8576
   • idba_2023_mediana: 450.1585
   • idba_2024_media: 682.9056
   • idba_2024_mediana: 646.1332
   • mu0_2021: 3.7529
   • mu0_2022: 0.0000
   • mu0_2023: 434.6111
   • mu0_2024: 648.1392

💾 Exportando archivos...
   ✅ Tabla principal: indicadores_cuadrantes_MANIZALES_2021-2024_20251014_152706.csv
   ✅ Priors por año: idba_priors_MANIZALES_2021-2024_20251014_152706.csv
   ✅ Met

## 📊 Resultados y Conclusiones

### Resumen del Análisis

Este notebook implementa la metodología **IDBA (Empirical Bayes) con conjugada Poisson-Gamma** para estimar densidades de muestras por cuadrante en Manizales (2021-2024), proporcionando estimaciones más robustas que los enfoques naïve tradicionales.

### Metodología Aplicada

1. **Prior Conjugado**: λᵢ ~ Gamma(α, β)
2. **Likelihood**: kᵢ|λᵢ ~ Poisson(λᵢ × Aᵢ)  
3. **Posterior**: λᵢ|kᵢ ~ Gamma(α + kᵢ, β + Aᵢ)
4. **Estimador EB**: E[λᵢ|kᵢ] = (α + kᵢ)/(β + Aᵢ)

### Ventajas del Enfoque IDBA

- **Shrinkage hacia el prior**: Reduce variabilidad en cuadrantes pequeños
- **Intervalos de credibilidad**: Cuantifica incertidumbre
- **Robustez estadística**: Maneja mejor cuadrantes con pocas observaciones
- **Interpretabilidad**: Los parámetros α y β tienen significado epidemiológico

### Interpretación de Resultados

- **Densidades altas**: Cuadrantes con actividad intensiva de muestreo
- **Shrinkage factor**: Indica confiabilidad (valores altos = menos confiable)
- **Intervalos de credibilidad**: Reflejan incertidumbre en las estimaciones
- **Ranking**: Permite priorización basada en evidencia estadística

### Archivos Generados

- **CSV principal**: Tabla completa con densidades naïve e IDBA
- **Metadatos JSON**: Parámetros del modelo y estadísticas resumen
- **Rankings y percentiles**: Para análisis comparativo

### Próximos Pasos

1. Validar resultados con conocimiento local del terreno
2. Comparar con otros períodos temporales
3. Integrar con análisis espaciales (clustering, autocorrelación)
4. Desarrollar mapas interactivos para visualización